# Notebook 14 — Ensemble sans fuite + lissage par patient

## Objectif
Battre le notebook 12 (OOF RMSE 4.07 / MAE 3.00) **avec une validation honnête**.

## Ce qui change par rapport au notebook 12
1. **Fuite corrigée** — dans le NB12, le target encoding de `rater_id` était calculé sur *tout* le train
   avant la CV (et utilisé par Optuna) : chaque fold de validation voyait sa propre cible. Ici l'encodage est
   calculé **dans chaque fold**, uniquement sur les patients d'entraînement (et en OOF interne pour les lignes d'entraînement).
2. **Features construites sur train + test réunis (X uniquement, jamais la cible)** — trajectoires patient
   et statistiques par évaluateur plus robustes, et identiques entre train et test.
3. **Biais des évaluateurs (sans cible)** — chaque `rater_id` a un biais systématique sur le score `off`
   par rapport à la trajectoire du patient. On l'estime, on « débiaise » `off`, puis on recalcule les trajectoires.
4. **Lissages par patient** — la cible est croissante dans le temps pour chaque patient et `off`/`on` en sont des
   mesures très bruitées : moyennes à noyau, régression isotone, signal combiné off+on.
5. **Ensemble de niveau 1** — LightGBM (cible directe, multi-seed), LightGBM sur le résidu `target - base`,
   LightGBM sur `log1p(target)` (bruit multiplicatif), CatBoost, XGBoost.
6. **Niveau 2 : lissage par patient** — la cible est un score OFF *corrigé*, donc lisse dans le temps pour un patient.
   Un second modèle utilise les prédictions OOF des autres visites du même patient (moyenne, pente, voisins, fit LOO).
   Valide car GroupKFold met toutes les visites d'un patient dans le même fold.
7. **Blend final** à poids positifs optimisés sur l'OOF, puis **projection monotone par patient** (isotone) et clip.

## Fichiers utilisés
- `../data/X_train.csv`, `../data/y_train.csv`, `../data/X_test.csv`

## Notebook suivant
`15_multi_models.ipynb` — ensemble multi-algorithmes final.

## 1. Imports et configuration

In [1]:
import json
import os
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

DATA_DIR = Path(os.environ.get("BOB_DATA_DIR", "../data"))
SUB_DIR = Path(os.environ.get("BOB_SUB_DIR", "../submissions"))

USE_SKORE = os.environ.get("BOB_USE_SKORE", "1") == "1"   # envoi du rapport sur Skore Hub
RUN_OPTUNA = False   # True = re-tuner LightGBM sur les features sans fuite (long)
FAST = os.environ.get("BOB_FAST", "0") == "1"             # True = test rapide de bout en bout

N_FOLDS = 5
FOLD_SEED = 42
LGB_SEEDS = [0] if FAST else [0, 1, 2]
N_ROUNDS = 300 if FAST else 6000
N_THREADS = os.cpu_count() or 4
TARGET_COL = "target"

USE_CATBOOST = True
USE_XGBOOST = True
try:
    from catboost import CatBoostRegressor
except ImportError:
    USE_CATBOOST = False
    print("catboost absent : modèle ignoré")
try:
    import xgboost as xgb
except ImportError:
    USE_XGBOOST = False
    print("xgboost absent : modèle ignoré")


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))


def isotonic_by_patient(frame, pred):
    """Projette les prédictions de chaque patient sur l'ensemble des suites croissantes avec l'âge.
    Sur le train, `target` ne décroît jamais au sein d'un patient : la projection ne peut que rapprocher
    les prédictions de la vérité (projection sur un convexe qui contient la vérité)."""
    from sklearn.isotonic import IsotonicRegression
    f = pd.DataFrame({"pid": frame["patient_id"].to_numpy(), "age": frame["age"].to_numpy(float),
                      "idx": frame["Index"].to_numpy(), "p": np.asarray(pred, float), "pos": np.arange(len(frame))})
    f = f.sort_values(["pid", "age", "idx"])
    out = np.empty(len(f))
    pids = f["pid"].to_numpy()
    bounds = np.flatnonzero(np.r_[True, pids[1:] != pids[:-1], True])
    p_sorted, a_sorted = f["p"].to_numpy(), f["age"].to_numpy()
    ir = IsotonicRegression(increasing=True)
    for s, e in zip(bounds[:-1], bounds[1:]):
        out[s:e] = p_sorted[s:e] if e - s == 1 else ir.fit_transform(a_sorted[s:e], p_sorted[s:e])
    res = np.empty(len(f))
    res[f["pos"].to_numpy()] = out
    return res


print(f"FAST={FAST}  seeds={LGB_SEEDS}  threads={N_THREADS}  catboost={USE_CATBOOST}  xgboost={USE_XGBOOST}")

FAST=False  seeds=[0, 1, 2]  threads=12  catboost=True  xgboost=True


## 2. Connexion Skore Hub

In [2]:
if USE_SKORE:
    import skore

    config = json.loads(Path("../.skore").read_text())
    os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
    skore.login()
    hub_project = skore.Project("bobathon-esilv", mode="hub", workspace=config["workspace"])
    print(f"Connecté : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté : workspace=BERNIE MADOFF


## 3. Chargement

On concatène `X_train` et `X_test` **sans la cible** pour construire les features patient et évaluateur.
Aucune information de `target` n'entre dans cette étape.

In [3]:
X_raw = pd.read_csv(DATA_DIR / "X_train.csv")
y_raw = pd.read_csv(DATA_DIR / "y_train.csv")
X_test_raw = pd.read_csv(DATA_DIR / "X_test.csv")

assert X_raw["Index"].is_unique and X_test_raw["Index"].is_unique
assert not set(X_raw["Index"]) & set(X_test_raw["Index"])

p_tr, p_te = set(X_raw["patient_id"]), set(X_test_raw["patient_id"])
overlap = p_tr & p_te
print(f"Train : {len(X_raw)} visites, {len(p_tr)} patients")
print(f"Test  : {len(X_test_raw)} visites, {len(p_te)} patients")
print(f"Patients communs train/test : {len(overlap)} "
      f"({X_test_raw['patient_id'].isin(overlap).mean():.1%} des visites test)")

X_all = pd.concat([X_raw.assign(_is_test=0), X_test_raw.assign(_is_test=1)], ignore_index=True)

Train : 44590 visites, 5576 patients
Test  : 11013 visites, 1395 patients
Patients communs train/test : 0 (0.0% des visites test)


## 4. Feature engineering

### 4.1 Trajectoires par patient (vectorisées)
Pour chaque patient et chaque score (`on`, `off`, `ledd`, puis `off` débiaisé) : pente, ordonnée, moyenne, écart-type,
première/dernière valeur, nombre de mesures, **prédiction leave-one-out** (fit linéaire sans la visite courante,
formule exacte via le levier `h_ii`) et fit quadratique.
La pente est légèrement régularisée (ridge) : avec des visites rapprochées, le fit linéaire non régularisé
du NB12 pouvait extrapoler à des valeurs absurdes (plusieurs centaines de points).

In [4]:
CAT = ["cohort", "gene", "rater_id"]
SLOPE_RIDGE = 1.0   # pénalité (en années²) sur la pente patient ; formule LOO exacte conservée


def _group_traj(age, v):
    """Features de trajectoire pour un patient. age, v : arrays 1D (NaN autorisés dans v)."""
    n = len(v)
    m = ~np.isnan(v)
    k = int(m.sum())
    out = {
        "slope": np.full(n, np.nan), "fit": np.full(n, np.nan), "loo": np.full(n, np.nan),
        "mean": np.full(n, np.nanmean(v) if k else np.nan),
        "std": np.full(n, np.nanstd(v) if k > 1 else np.nan),
        "first": np.full(n, v[m][0] if k else np.nan),
        "last": np.full(n, v[m][-1] if k else np.nan),
        "cnt": np.full(n, k, dtype=float),
        "quad": np.full(n, np.nan),
    }
    if k >= 1:
        # LOO de la moyenne (repli quand la pente n'est pas estimable)
        s = v[m].sum()
        loo_mean = np.full(n, s / k)
        if k > 1:
            loo_mean[m] = (s - v[m]) / (k - 1)
        else:
            loo_mean[m] = np.nan
        out["loo_mean"] = loo_mean
    else:
        out["loo_mean"] = np.full(n, np.nan)
    if k >= 2:
        xs, ys = age[m], v[m]
        xm, ym = xs.mean(), ys.mean()
        sxx = ((xs - xm) ** 2).sum() + SLOPE_RIDGE      # pente « ridge » : évite les extrapolations folles
        if sxx > 1e-9:
            b = ((xs - xm) * (ys - ym)).sum() / sxx
            a = ym - b * xm
            fit = a + b * age
            out["slope"][:] = b
            out["fit"] = fit
            loo = fit.copy()                       # visites sans mesure : fit complet
            h = 1.0 / k + (xs - xm) ** 2 / sxx
            with np.errstate(divide="ignore", invalid="ignore"):
                loo_m = (fit[m] - h * ys) / (1 - h)
            loo_m[(1 - h) < 1e-9] = np.nan
            loo[m] = loo_m
            out["loo"] = loo
    if k >= 4 and np.ptp(age[m]) > 1e-9:
        c = np.polyfit(age[m] - age.mean(), v[m], 2)
        out["quad"] = np.polyval(c, age - age.mean())
    return out


def add_patient_traj(d, col, pre):
    """d doit être trié par patient_id, age. Ajoute les colonnes {pre}_*."""
    res = {}
    ages = d["age"].to_numpy(float)
    vals = d[col].to_numpy(float)
    bounds = np.flatnonzero(np.r_[True, d["patient_id"].to_numpy()[1:] != d["patient_id"].to_numpy()[:-1], True])
    for s, e in zip(bounds[:-1], bounds[1:]):
        o = _group_traj(ages[s:e], vals[s:e])
        for kk, arr in o.items():
            res.setdefault(kk, []).append(arr)
    for kk, arrs in res.items():
        d[f"{pre}_{kk}"] = np.concatenate(arrs)
    # LOO linéaire si dispo, sinon LOO moyenne
    d[f"{pre}_loo2"] = d[f"{pre}_loo"].fillna(d[f"{pre}_loo_mean"])
    return d


print("Fonctions de trajectoire définies")

Fonctions de trajectoire définies


### 4.2 Lissages par patient : noyau gaussien et régression isotone
La cible est **croissante** dans le temps pour chaque patient (vérifié plus bas) et `off`/`on` sont des mesures
très bruitées de cette courbe (bruit à peu près proportionnel au niveau). On débruite donc chaque patient :
- moyennes à noyau gaussien sur l'âge (largeurs 0.75, 1.5, 3 ans), avec et sans la visite courante ;
- régression isotone croissante (avec et sans la visite courante) ;
- sur `off` débiaisé, `on` débiaisé, et un signal combiné `comb` = moyenne de `off` et `on` remis à l'échelle de `off`
  (rapport médian off/on calculé sans la cible), ce qui double le nombre de mesures par patient.

In [5]:
from sklearn.isotonic import IsotonicRegression

def add_smooth_feats(d, col, pre, bws=(0.75, 1.5, 3.0)):
    """d trié patient, age. Lissages noyau (avec et sans la visite courante) + isotonique croissante."""
    ages = d["age"].to_numpy(float); vals = d[col].to_numpy(float)
    pids = d["patient_id"].to_numpy()
    bounds = np.flatnonzero(np.r_[True, pids[1:] != pids[:-1], True])
    outs = {f"{pre}_k{bw}": np.full(len(d), np.nan) for bw in bws}
    outs.update({f"{pre}_k{bw}_loo": np.full(len(d), np.nan) for bw in bws})
    outs[f"{pre}_iso"] = np.full(len(d), np.nan)
    outs[f"{pre}_iso_loo"] = np.full(len(d), np.nan)
    ir = IsotonicRegression(increasing=True, out_of_bounds="clip")
    for s, e in zip(bounds[:-1], bounds[1:]):
        a, v = ages[s:e], vals[s:e]
        m = ~np.isnan(v)
        if m.sum() == 0: continue
        D = a[:, None] - a[None, m]
        for bw in bws:
            W = np.exp(-0.5 * (D / bw) ** 2)
            num, den = W @ v[m], W.sum(1)
            outs[f"{pre}_k{bw}"][s:e] = num / den
            # loo : retirer la contribution propre
            self_w = np.zeros(e - s); self_w[m] = 1.0
            selfv = np.where(m, v, 0.0)
            den2 = den - self_w
            with np.errstate(invalid="ignore", divide="ignore"):
                outs[f"{pre}_k{bw}_loo"][s:e] = np.where(den2 > 1e-6, (num - selfv) / den2, np.nan)
        if m.sum() >= 2:
            ir.fit(a[m], v[m]); outs[f"{pre}_iso"][s:e] = ir.predict(a)
        if m.sum() >= 3:
            loo = np.full(e - s, np.nan); idx = np.flatnonzero(m)
            for j in range(e - s):
                keep = m.copy(); keep[j] = False
                ir.fit(a[keep], v[keep]); loo[j] = ir.predict(a[j:j+1])[0]
            outs[f"{pre}_iso_loo"][s:e] = loo
        elif m.sum() >= 1:
            outs[f"{pre}_iso_loo"][s:e] = np.nanmean(v[m])
    for k, arr in outs.items(): d[k] = arr
    return d

def add_combined(d):
    # échelle on -> off estimée sans cible (rapport médian off/on sur les visites qui ont les deux)
    both = d["off_db"].notna() & d["on_db"].notna()
    k = float(np.median(d.loc[both, "off_db"] / d.loc[both, "on_db"].clip(lower=1)))
    d["comb"] = d[["off_db"]].assign(o=d["on_db"] * k).mean(axis=1)
    d["comb_n"] = d["off_db"].notna().astype(int) + d["on_db"].notna().astype(int)
    d["log_off"] = np.log1p(d["off_db"].clip(lower=0))
    return d, k


print("Fonctions de lissage définies")

Fonctions de lissage définies


### 4.3 Construction complète (train + test, sans cible)

In [6]:
def build_all(X):
    d = X.sort_values(["patient_id", "age", "Index"]).reset_index(drop=True).copy()
    gb = d.groupby("patient_id", sort=False)

    d["dx_missing"] = d["age_at_diagnosis"].isna().astype(int)
    d["dur"] = d["age"] - d["age_at_diagnosis"]
    d["dur_first"] = gb["dur"].transform("min")
    d["age_first"] = gb["age"].transform("min")
    d["t_rel"] = d["age"] - d["age_first"]
    d["t_span"] = gb["age"].transform("max") - d["age_first"]
    d["n_vis"] = gb["age"].transform("size")
    d["vis_idx"] = gb.cumcount()
    d["vis_frac"] = d["vis_idx"] / (d["n_vis"] - 1).clip(lower=1)
    d["gap_prev"] = gb["age"].diff()
    d["gap_next"] = -gb["age"].diff(-1)

    d["on_off_diff"] = d["off"] - d["on"]
    d["on_off_ratio"] = d["on"] / d["off"].replace(0, np.nan)
    d["ledd_per_dur"] = d["ledd"] / d["dur"].clip(lower=0.5)
    d["has_on"] = d["on"].notna().astype(int)
    d["has_off"] = d["off"].notna().astype(int)
    d["has_ledd"] = d["ledd"].notna().astype(int)
    d["tsi_off_dev"] = d["time_since_intake_off"] - 12
    d["tsi_off_missing"] = d["time_since_intake_off"].isna().astype(int)
    d["on_x_ledd"] = d["on"] * d["ledd"]
    d["off_x_ledd"] = d["off"] * d["ledd"]
    d["age_sq"] = d["age"] ** 2
    d["dur_sq"] = d["dur"] ** 2

    for c in ["on", "off", "ledd", "time_since_intake_off", "time_since_intake_on"]:
        for k in (1, 2):
            d[f"{c}_lag{k}"] = gb[c].shift(k)
            d[f"{c}_lead{k}"] = gb[c].shift(-k)
    d["off_nb_mean"] = d[["off_lag1", "off_lead1"]].mean(axis=1)
    d["on_nb_mean"] = d[["on_lag1", "on_lead1"]].mean(axis=1)
    d["off_delta_prev"] = d["off"] - d["off_lag1"]
    d["off_delta_next"] = d["off_lead1"] - d["off"]

    for c in ["on", "off", "ledd"]:
        d = add_patient_traj(d, c, c)

    d["pseudo_off_from_on"] = d["on"] * (d["off_mean"] / d["on_mean"])
    d["traj_blend"] = d[["off_loo", "pseudo_off_from_on"]].mean(axis=1)
    d["p_age_dx"] = gb["age_at_diagnosis"].transform("first")
    d["off_loo_ratio"] = d["off_loo"] / d["off"].replace(0, np.nan)
    d["off_loo_resid"] = d["off"] - d["off_loo2"]
    d["on_loo_resid"] = d["on"] - d["on_loo2"]

    # --- Biais des évaluateurs (sans cible) : écart moyen off - trajectoire LOO, avec shrinkage ---
    K_SHRINK = 10.0
    for c in ["off", "on"]:
        r = d[f"{c}_loo_resid"]
        g = r.groupby(d["rater_id"])
        s, n = g.transform("sum"), g.transform("count")
        d[f"rater_{c}_bias"] = (s / (n + K_SHRINK)).fillna(0.0)
    d["rater_n"] = d.groupby("rater_id")["Index"].transform("size").fillna(0)
    d["rater_n_patients"] = d.groupby("rater_id")["patient_id"].transform("nunique").fillna(0)
    d["rater_missing"] = d["rater_id"].isna().astype(int)

    # Biais selon le délai depuis la prise (sans cible)
    tsi_bin = pd.cut(d["time_since_intake_off"], bins=[-np.inf, 2, 4, 6, 8, 10, 12, 14, np.inf])
    d["tsi_off_bias"] = d.groupby(tsi_bin, observed=True)["off_loo_resid"].transform("mean")

    # off débiaisé de l'évaluateur, puis nouvelles trajectoires
    d["off_db"] = d["off"] - d["rater_off_bias"]
    d["on_db"] = d["on"] - d["rater_on_bias"]
    d = add_patient_traj(d, "off_db", "offdb")
    d = add_patient_traj(d, "on_db", "ondb")
    d["offdb_loo_resid"] = d["off_db"] - d["offdb_loo2"]
    d["offdb_nb_mean"] = pd.concat(
        [d.groupby("patient_id", sort=False)["off_db"].shift(1),
         d.groupby("patient_id", sort=False)["off_db"].shift(-1)], axis=1).mean(axis=1)

    # Signal combiné off/on et lissages
    d, k_on = add_combined(d)
    d = add_patient_traj(d, "comb", "comb")
    for c, pre in [("off_db", "sdb"), ("on_db", "son"), ("comb", "scomb")]:
        d = add_smooth_feats(d, c, pre)

    # Base pour le modèle sur résidu : meilleure estimation « naïve » disponible
    d["base"] = (d["scomb_k1.5"]
                 .fillna(d["comb_loo2"])
                 .fillna(d["offdb_loo2"])
                 .fillna(d["pseudo_off_from_on"]))
    d["base"] = d["base"].fillna(d["base"].median()).clip(0, 120)

    for c in CAT:
        d[c] = d[c].astype("category")
    return d


t0 = time.time()
full = build_all(X_all)
print(f"Features construites en {time.time() - t0:.1f}s — shape {full.shape}")

d = (full[full["_is_test"] == 0].merge(y_raw, on="Index")
     .sort_values("Index").reset_index(drop=True))
dt = (full[full["_is_test"] == 1].set_index("Index")
      .loc[X_test_raw["Index"]].reset_index())       # ordre exact de X_test
assert dt["Index"].equals(X_test_raw["Index"])
assert len(d) == len(X_raw)

_s = d.sort_values(["patient_id", "age", "Index"])
_dec = (_s.groupby("patient_id")[TARGET_COL].diff() < -1e-9).sum()
print(f"Visites où la cible baisse par rapport à la visite précédente du patient : {_dec}")

y = d[TARGET_COL].to_numpy(float)
groups = d["patient_id"].to_numpy()
BASE_COLS = [c for c in d.columns if c not in {"Index", "patient_id", TARGET_COL, "_is_test"}]
print(f"Train {d.shape}  Test {dt.shape}  Nb features {len(BASE_COLS)}")
print(f"RMSE de la base naïve (sans modèle) : {rmse(y, d['base']):.4f}  MAE {mae(y, d['base']):.4f}")

Features construites en 167.0s — shape (55603, 171)
Visites où la cible baisse par rapport à la visite précédente du patient : 0
Train (44590, 172)  Test (11013, 171)  Nb features 168
RMSE de la base naïve (sans modèle) : 8.9416  MAE 6.9084


## 5. Folds par patient (fixes pour tous les modèles)

Les mêmes folds servent au niveau 1, au niveau 2 et au blend : c'est ce qui rend l'empilement valide.

In [7]:
rng = np.random.default_rng(FOLD_SEED)
uniq = np.unique(groups)
perm = rng.permutation(len(uniq))
pat_fold = dict(zip(uniq[perm], np.arange(len(uniq)) % N_FOLDS))
fold_id = np.array([pat_fold[g] for g in groups])
FOLDS = [(np.flatnonzero(fold_id != k), np.flatnonzero(fold_id == k)) for k in range(N_FOLDS)]
for k, (tr, va) in enumerate(FOLDS):
    assert not set(groups[tr]) & set(groups[va])
print("Taille des folds :", [len(va) for _, va in FOLDS])

Taille des folds : [8737, 9032, 8949, 8942, 8930]


## 6. Target encoding de `rater_id` **dans le fold**

On encode le biais de l'évaluateur sur le résidu `target - base` (moyenne lissée).
- lignes de validation / test : encodage appris sur les lignes d'entraînement du fold ;
- lignes d'entraînement : encodage OOF interne (GroupKFold par patient), pour que le modèle
  ne voie jamais une valeur calculée avec sa propre cible.

In [8]:
TE_SMOOTH = 30.0


def _te_fit(keys, resid):
    g = pd.DataFrame({"k": keys, "r": resid}).groupby("k", observed=True)["r"].agg(["sum", "count"])
    return g["sum"] / (g["count"] + TE_SMOOTH)


def te_feature(tr_idx, apply_frames, inner_splits=5):
    """Retourne (te_train, [te pour chaque frame de apply_frames])."""
    keys_tr = d["rater_id"].astype(object).fillna("__NA__").astype(str).to_numpy()[tr_idx]
    res_tr = (y - d["base"].to_numpy())[tr_idx]
    te_tr = np.zeros(len(tr_idx))
    for itr, iva in GroupKFold(n_splits=inner_splits).split(tr_idx, groups=groups[tr_idx]):
        enc = _te_fit(keys_tr[itr], res_tr[itr])
        te_tr[iva] = pd.Series(keys_tr[iva]).map(enc).fillna(0.0).to_numpy()
    enc = _te_fit(keys_tr, res_tr)
    outs = [frame["rater_id"].astype(object).fillna("__NA__").astype(str).map(enc).fillna(0.0).to_numpy() for frame in apply_frames]
    return te_tr, outs


COLS = BASE_COLS + ["rater_te"]
print(f"Nb features (avec TE) : {len(COLS)}")

Nb features (avec TE) : 169


## 7. Niveau 1 — modèles de base

Chaque modèle renvoie ses prédictions OOF (train) et la moyenne des modèles de fold sur le test.
Les paramètres LightGBM reprennent ceux trouvés par Optuna dans le NB12 (option `RUN_OPTUNA` pour re-tuner).

In [9]:
LGB_PARAMS = dict(
    objective="regression", learning_rate=0.0144, num_leaves=74, min_child_samples=16,
    feature_fraction=0.515, bagging_fraction=0.635, bagging_freq=1, lambda_l2=0.373,
    cat_smooth=42, verbose=-1, num_threads=N_THREADS,
)

if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        p = dict(LGB_PARAMS,
                 learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                 num_leaves=trial.suggest_int("num_leaves", 15, 128),
                 min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
                 feature_fraction=trial.suggest_float("feature_fraction", 0.3, 1.0),
                 bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
                 lambda_l2=trial.suggest_float("lambda_l2", 0.01, 30.0, log=True),
                 min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.5),
                 cat_smooth=trial.suggest_int("cat_smooth", 5, 80))
        errs = []
        for tr, va in FOLDS[:3]:
            te_tr, (te_va,) = te_feature(tr, [d.iloc[va]])
            Xtr = d.iloc[tr][BASE_COLS].assign(rater_te=te_tr)
            Xva = d.iloc[va][BASE_COLS].assign(rater_te=te_va)
            m = lgb.train(p, lgb.Dataset(Xtr, y[tr]), N_ROUNDS,
                          valid_sets=[lgb.Dataset(Xva, y[va])],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
            errs.append(rmse(y[va], m.predict(Xva, num_iteration=m.best_iteration)))
        return float(np.mean(errs))

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=40, show_progress_bar=True)
    LGB_PARAMS.update(study.best_params)
    print(f"Optuna : {study.best_value:.4f}", study.best_params)

In [10]:
# Matrices par fold (TE dans le fold) — calculées une fois, réutilisées par tous les modèles
FOLD_DATA = []
for k, (tr, va) in enumerate(FOLDS):
    te_tr, (te_va, te_te) = te_feature(tr, [d.iloc[va], dt])
    FOLD_DATA.append(dict(
        tr=tr, va=va,
        Xtr=d.iloc[tr][BASE_COLS].assign(rater_te=te_tr),
        Xva=d.iloc[va][BASE_COLS].assign(rater_te=te_va),
        Xte=dt[BASE_COLS].assign(rater_te=te_te),
    ))
base_tr = d["base"].to_numpy(float)
base_te = dt["base"].to_numpy(float)
L1 = {}   # nom -> (oof, test)


def report(name, oof):
    print(f"  {name:<14s} OOF RMSE={rmse(y, oof):.4f}  MAE={mae(y, oof):.4f}")


def run_lgb(name, params, mode="direct", seeds=LGB_SEEDS):
    """mode : 'direct' (cible brute), 'resid' (cible - base), 'log' (log1p de la cible)."""
    fwd = {"direct": lambda t, b: t, "resid": lambda t, b: t - b, "log": lambda t, b: np.log1p(t)}[mode]
    inv = {"direct": lambda p, b: p, "resid": lambda p, b: p + b, "log": lambda p, b: np.expm1(p)}[mode]
    oof, pte = np.zeros(len(d)), np.zeros(len(dt))
    t0 = time.time()
    for fd in FOLD_DATA:
        tr, va = fd["tr"], fd["va"]
        for s in seeds:
            p = dict(params, seed=s, bagging_seed=s, feature_fraction_seed=s)
            m = lgb.train(p, lgb.Dataset(fd["Xtr"], fwd(y[tr], base_tr[tr])), N_ROUNDS,
                          valid_sets=[lgb.Dataset(fd["Xva"], fwd(y[va], base_tr[va]))],
                          callbacks=[lgb.early_stopping(150, verbose=False)])
            oof[va] += inv(m.predict(fd["Xva"], num_iteration=m.best_iteration), base_tr[va]) / len(seeds)
            pte += inv(m.predict(fd["Xte"], num_iteration=m.best_iteration), base_te) / (len(seeds) * N_FOLDS)
    L1[name] = (oof, pte)
    report(name, oof)
    print(f"    ({time.time() - t0:.0f}s)")


run_lgb("lgb_l2", LGB_PARAMS)

  lgb_l2         OOF RMSE=3.5793  MAE=2.6019
    (926s)


In [11]:
run_lgb("lgb_resid", LGB_PARAMS, mode="resid")

  lgb_resid      OOF RMSE=3.5842  MAE=2.6183
    (25486s)


In [12]:
run_lgb("lgb_log", LGB_PARAMS, mode="log")

  lgb_log        OOF RMSE=3.6017  MAE=2.5954
    (814s)


In [13]:
if USE_CATBOOST:
    oof, pte = np.zeros(len(d)), np.zeros(len(dt))
    t0 = time.time()

    def _cb_frame(X):
        X = X.copy()
        for c in CAT:
            X[c] = X[c].astype(object).fillna("__NA__").astype(str)
        return X

    for fd in FOLD_DATA:
        tr, va = fd["tr"], fd["va"]
        m = CatBoostRegressor(
            iterations=N_ROUNDS, learning_rate=0.04 if not FAST else 0.1, depth=8,
            l2_leaf_reg=3, loss_function="RMSE", random_seed=0, od_type="Iter", od_wait=200,
            thread_count=N_THREADS, verbose=0,
        )
        m.fit(_cb_frame(fd["Xtr"]), y[tr] - base_tr[tr], cat_features=CAT,
              eval_set=(_cb_frame(fd["Xva"]), y[va] - base_tr[va]), use_best_model=True)
        oof[va] = m.predict(_cb_frame(fd["Xva"])) + base_tr[va]
        pte += (m.predict(_cb_frame(fd["Xte"])) + base_te) / N_FOLDS
    L1["catboost"] = (oof, pte)
    report("catboost", oof)
    print(f"    ({time.time() - t0:.0f}s)")

  catboost       OOF RMSE=3.5970  MAE=2.6342
    (4039s)


In [14]:
if USE_XGBOOST:
    oof, pte = np.zeros(len(d)), np.zeros(len(dt))
    t0 = time.time()
    xgb_params = dict(
        objective="reg:squarederror", eta=0.02 if not FAST else 0.1, max_depth=7, min_child_weight=5,
        subsample=0.7, colsample_bytree=0.5, reg_lambda=2.0, tree_method="hist",
        max_cat_to_onehot=1, nthread=N_THREADS, seed=0,
    )
    for fd in FOLD_DATA:
        tr, va = fd["tr"], fd["va"]
        dtr = xgb.DMatrix(fd["Xtr"], y[tr] - base_tr[tr], enable_categorical=True)
        dva = xgb.DMatrix(fd["Xva"], y[va] - base_tr[va], enable_categorical=True)
        dte_m = xgb.DMatrix(fd["Xte"], enable_categorical=True)
        m = xgb.train(xgb_params, dtr, N_ROUNDS, evals=[(dva, "va")],
                      early_stopping_rounds=200, verbose_eval=False)
        rng_it = (0, m.best_iteration + 1)
        oof[va] = m.predict(dva, iteration_range=rng_it) + base_tr[va]
        pte += (m.predict(dte_m, iteration_range=rng_it) + base_te) / N_FOLDS
    L1["xgboost"] = (oof, pte)
    report("xgboost", oof)
    print(f"    ({time.time() - t0:.0f}s)")

  xgboost        OOF RMSE=3.6357  MAE=2.6618
    (989s)


## 8. Blend niveau 1 (poids positifs, somme = 1, optimisés sur l'OOF)

In [15]:
def fit_blend(P, y, loss="rmse"):
    n = P.shape[1]
    f = (lambda w: rmse(y, P @ w)) if loss == "rmse" else (lambda w: mae(y, P @ w))
    res = minimize(f, np.full(n, 1 / n), method="SLSQP", bounds=[(0, 1)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}])
    return res.x


names1 = list(L1)
P1 = np.column_stack([L1[n][0] for n in names1])
T1 = np.column_stack([L1[n][1] for n in names1])
w1 = fit_blend(P1, y)
blend1_oof, blend1_te = P1 @ w1, T1 @ w1
print("Poids :", dict(zip(names1, np.round(w1, 3))))
report("blend_L1", blend1_oof)
report("blend_L1+iso", isotonic_by_patient(d, blend1_oof))

Poids : {'lgb_l2': np.float64(0.383), 'lgb_resid': np.float64(0.211), 'lgb_log': np.float64(0.064), 'catboost': np.float64(0.327), 'xgboost': np.float64(0.014)}
  blend_L1       OOF RMSE=3.5303  MAE=2.5712
  blend_L1+iso   OOF RMSE=3.5255  MAE=2.5679


## 9. Niveau 2 — lissage des prédictions par patient

Pour chaque visite, on regarde les prédictions OOF des **autres visites du même patient** :
moyenne, écart-type, pente sur l'âge, prédiction leave-one-out, voisins précédent/suivant.
Toutes ces prédictions viennent de modèles qui n'ont jamais vu ce patient (même fold), donc pas de fuite.
Côté test, on applique la même transformation aux prédictions test du niveau 1.

In [16]:
L2_RAW = ["age", "t_rel", "n_vis", "vis_frac", "gap_prev", "gap_next", "off", "on", "off_db",
          "offdb_loo2", "offdb_mean", "time_since_intake_off", "time_since_intake_on", "ledd",
          "rater_off_bias", "rater_n", "has_off", "has_on", "base", "dur",
          "scomb_k1.5", "scomb_iso", "sdb_iso", "comb_n", "comb_cnt"]


def level2_frame(frame, preds_matrix, blend):
    f = frame[["Index", "patient_id", "age"] + [c for c in L2_RAW if c != "age"]].copy()
    for j, n in enumerate(names1):
        f[f"p_{n}"] = preds_matrix[:, j]
    f["p_blend"] = blend
    f["p_spread"] = preds_matrix.std(axis=1)
    f = f.sort_values(["patient_id", "age", "Index"]).reset_index(drop=True)
    f = add_patient_traj(f, "p_blend", "pb")
    g = f.groupby("patient_id", sort=False)["p_blend"]
    for k in (1, 2):
        f[f"pb_lag{k}"] = g.shift(k)
        f[f"pb_lead{k}"] = g.shift(-k)
    f["pb_nb_mean"] = f[["pb_lag1", "pb_lead1"]].mean(axis=1)
    f["pb_dev_fit"] = f["p_blend"] - f["pb_fit"]
    f["pb_dev_loo"] = f["p_blend"] - f["pb_loo2"]
    f["pb_dev_mean"] = f["p_blend"] - f["pb_loo_mean"]
    return f.set_index("Index").loc[frame["Index"]].reset_index()


F2 = level2_frame(d, P1, blend1_oof)
F2te = level2_frame(dt, T1, blend1_te)
COLS2 = [c for c in F2.columns if c not in {"Index", "patient_id"}]
print(f"Features niveau 2 : {len(COLS2)}")

L2_PARAMS = dict(objective="regression", learning_rate=0.02, num_leaves=15, min_child_samples=40,
                 feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                 verbose=-1, num_threads=N_THREADS)
L2 = {}
for nm, obj in [("l2_lgb", "regression"), ("l2_huber", "huber")]:
    oof2, pte2 = np.zeros(len(d)), np.zeros(len(dt))
    for tr, va in FOLDS:
        for s in LGB_SEEDS:
            p = dict(L2_PARAMS, objective=obj, alpha=4.0, seed=s)
            r_tr, r_va = blend1_oof[tr], blend1_oof[va]     # on prédit la correction du blend L1
            m = lgb.train(p, lgb.Dataset(F2.iloc[tr][COLS2], y[tr] - r_tr), N_ROUNDS,
                          valid_sets=[lgb.Dataset(F2.iloc[va][COLS2], y[va] - r_va)],
                          callbacks=[lgb.early_stopping(150, verbose=False)])
            oof2[va] += (m.predict(F2.iloc[va][COLS2], num_iteration=m.best_iteration) + r_va) / len(LGB_SEEDS)
            pte2 += (m.predict(F2te[COLS2], num_iteration=m.best_iteration) + blend1_te) / (len(LGB_SEEDS) * N_FOLDS)
    L2[nm] = (oof2, pte2)
    report(nm, oof2)

Features niveau 2 : 51
  l2_lgb         OOF RMSE=3.4991  MAE=2.5487
  l2_huber       OOF RMSE=3.4998  MAE=2.5334


## 10. Blend final et choix

In [17]:
names_f = names1 + list(L2)
Pf = np.column_stack([P1] + [L2[n][0] for n in L2])
Tf = np.column_stack([T1] + [L2[n][1] for n in L2])
wf = fit_blend(Pf, y)
final_oof, final_te = Pf @ wf, Tf @ wf
print("Poids finaux :", {n: round(float(w), 3) for n, w in zip(names_f, wf) if w > 1e-3})

report("blend L1+L2", final_oof)
blend_oof_raw = final_oof.copy()

# Monotonie par patient (la cible ne décroît jamais dans le temps), puis clip sur la plage observée
final_oof = isotonic_by_patient(d, final_oof)
final_te = isotonic_by_patient(dt, final_te)
y_lo, y_hi = float(y.min()), float(y.max())
final_oof = np.clip(final_oof, y_lo, y_hi)
final_te = np.clip(final_te, y_lo, y_hi)
report("FINAL (+iso)", final_oof)

print("\nPar fold :")
for k, (_, va) in enumerate(FOLDS):
    print(f"  Fold {k + 1} — RMSE={rmse(y[va], final_oof[va]):.4f}  MAE={mae(y[va], final_oof[va]):.4f}")

Poids finaux : {'xgboost': 0.01, 'l2_lgb': 0.481, 'l2_huber': 0.509}
  blend L1+L2    OOF RMSE=3.4974  MAE=2.5394
  FINAL (+iso)   OOF RMSE=3.4943  MAE=2.5369

Par fold :
  Fold 1 — RMSE=3.3455  MAE=2.4483
  Fold 2 — RMSE=3.5044  MAE=2.5604
  Fold 3 — RMSE=3.4511  MAE=2.5071
  Fold 4 — RMSE=3.5469  MAE=2.5643
  Fold 5 — RMSE=3.6142  MAE=2.6022


## 11. Soumission

In [18]:
submission = X_test_raw[["Index"]].copy()
submission["target"] = final_te

assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(X_test_raw)
assert submission["Index"].equals(X_test_raw["Index"])
assert submission["target"].notna().all()

SUB_DIR.mkdir(exist_ok=True, parents=True)
submission.to_csv(SUB_DIR / "14_ensemble_l2.csv", index=False)
print(f"Soumission : {SUB_DIR / '14_ensemble_l2.csv'} ({len(submission)} lignes)")
print(f"Min {final_te.min():.2f}  Max {final_te.max():.2f}  Mean {final_te.mean():.2f}")
display(submission.head())

Soumission : ..\submissions\14_ensemble_l2.csv (11013 lignes)
Min 0.00  Max 93.85  Mean 36.63


,Index,target
0,66,20.449622
1,67,25.967915
2,68,26.771081
3,69,29.818810
4,70,30.900434


## 12. Envoi Skore Hub

`skore.evaluate` attend un estimateur scikit-learn : on y envoie le LightGBM principal sur les features
sans cible (le TE de `rater_id` est exclu car il doit être recalculé dans chaque fold), avec les mêmes folds.
Les scores de l'ensemble complet sont ceux de la section 10.

In [19]:
if USE_SKORE:
    from lightgbm import LGBMRegressor

    lgbm_skore = LGBMRegressor(
        n_estimators=2000, learning_rate=LGB_PARAMS["learning_rate"], num_leaves=LGB_PARAMS["num_leaves"],
        min_child_samples=LGB_PARAMS["min_child_samples"], colsample_bytree=LGB_PARAMS["feature_fraction"],
        subsample=LGB_PARAMS["bagging_fraction"], subsample_freq=1, reg_lambda=LGB_PARAMS["lambda_l2"],
        cat_smooth=LGB_PARAMS["cat_smooth"], random_state=0, verbose=-1,
    )
    report_14 = skore.evaluate(lgbm_skore, d[BASE_COLS], d[TARGET_COL], splitter=FOLDS)
    hub_project.put("14_ensemble_l2", report_14)
    print("Rapport Hub envoyé : 14_ensemble_l2")
    print(f"RMSE skore (LGBM seul) : {float(report_14.metrics.rmse().iloc[:, 0].values.mean()):.4f}")

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/72941

Rapport Hub envoyé : 14_ensemble_l2
RMSE skore (LGBM seul) : 3.6063


## 13. Récapitulatif

In [20]:
rows = [{"Modèle": "LightGBM Optuna NB12 (TE avec fuite)", "RMSE": 4.0711, "MAE": 3.0027},
        {"Modèle": "Base naïve (lissage off/on)", "RMSE": rmse(y, d["base"]), "MAE": mae(y, d["base"])}]
rows += [{"Modèle": f"L1 {n}", "RMSE": rmse(y, L1[n][0]), "MAE": mae(y, L1[n][0])} for n in L1]
rows += [{"Modèle": "Blend L1", "RMSE": rmse(y, blend1_oof), "MAE": mae(y, blend1_oof)}]
rows += [{"Modèle": f"L2 {n}", "RMSE": rmse(y, L2[n][0]), "MAE": mae(y, L2[n][0])} for n in L2]
rows += [{"Modèle": "Blend L1+L2", "RMSE": rmse(y, blend_oof_raw), "MAE": mae(y, blend_oof_raw)}]
rows += [{"Modèle": "FINAL (+ monotonie patient, clip)", "RMSE": rmse(y, final_oof), "MAE": mae(y, final_oof)}]
recap = pd.DataFrame(rows)
display(recap.round(4))

,Modèle,RMSE,MAE
0,LightGBM Optuna NB12 (TE avec fuite),4.0711,3.0027
1,Base naïve (lissage off/on),8.9416,6.9084
2,L1 lgb_l2,3.5793,2.6019
3,L1 lgb_resid,3.5842,2.6183
4,L1 lgb_log,3.6017,2.5954
5,L1 catboost,3.5970,2.6342
6,L1 xgboost,3.6357,2.6618
7,Blend L1,3.5303,2.5712
8,L2 l2_lgb,3.4991,2.5487
9,L2 l2_huber,3.4998,2.5334
